# Forge on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NU-Deep-Learning/Forge/blob/main/scripts/run_colab.ipynb)

Run training, distillation, and RL experiments on Google Colab.

**One-time setup** - add tokens to Colab Secrets (key icon in left sidebar):
- `GITHUB_TOKEN` - [github.com/settings/tokens](https://github.com/settings/tokens) (repo scope)
- `HF_TOKEN` - [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (for gated models like Llama)
- `WANDB_API_KEY` - [wandb.ai/authorize](https://wandb.ai/authorize) (optional, for experiment tracking)

**Hardware:** A100 recommended. T4/V100 work for BERT-size models.

## 1. Setup

In [ ]:
import os

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    # Clone repo
    if os.path.exists('/content/Forge'):
        %cd /content/Forge
        !git pull
    else:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/NU-Deep-Learning/Forge.git /content/Forge
        %cd /content/Forge

    # Install deps
    !pip install -q uv
    !uv pip install -q -e . --system
    print('Setup complete')
else:
    print('Running locally')

In [ ]:
# Service logins
import os

if IN_COLAB:
    from google.colab import userdata

    # W&B (optional)
    try:
        api_key = userdata.get('WANDB_API_KEY')
        if api_key:
            os.environ['WANDB_API_KEY'] = api_key
            import wandb
            wandb.login(key=api_key, verify=True)
            print('W&B: logged in')
        else:
            raise ValueError('empty key')
    except Exception:
        os.environ['WANDB_MODE'] = 'disabled'
        print('W&B: disabled (no key or invalid)')

    # HuggingFace (for gated models)
    try:
        from huggingface_hub import login
        login(token=userdata.get('HF_TOKEN'))
        print('HF: logged in')
    except Exception:
        print('HF: not logged in (add HF_TOKEN for Llama models)')

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Run Training

Pick a config and run. Use `--max_steps N` for quick tests.

**Example configs:**

| Task | Config | Time (full) |
|------|--------|-------------|
| MNLI (SFT) | `configs/classification/mnli/distilbert/00_sft.yaml` | ~1-2h |
| MNLI (KD) | `configs/classification/mnli/distilbert/bert-base/ablations/01_kd_frozen-base.yaml` | ~1-2h |
| QQP (SFT) | `configs/classification/qqp/distilbert/00_sft.yaml` | ~2-3h |
| Translation | `configs/translation/europarl/llama32-1b/00_sft.yaml` | ~6-8h |

GPU-optimized configs live under `configs/.../gpu/t4/` and `configs/.../gpu/a100/`.

Browse all configs: `!find configs -name '*.yaml' | head -20`

In [ ]:
# Quick test (~15 min)
!python forge.py \
    --config configs/classification/mnli/distilbert/00_sft.yaml \
    --max_steps 500

In [ ]:
# Full run - uncomment one:

# MNLI SFT (T4-optimized)
# !python forge.py --config configs/classification/mnli/distilbert/gpu/t4/batch128_lr1p6e-4.yaml

# MNLI KD (frozen teacher)
# !python forge.py --config configs/classification/mnli/distilbert/bert-base/ablations/01_kd_frozen-base.yaml

# QQP SFT
# !python forge.py --config configs/classification/qqp/distilbert/00_sft.yaml

# Translation (requires HF_TOKEN for Llama)
# !python forge.py --config configs/translation/europarl/llama32-1b/00_sft.yaml

## 3. Results

In [ ]:
# Find and display training results
import json
from pathlib import Path

state_files = sorted(Path('outputs').rglob('training_state.json')) if Path('outputs').exists() else []
if state_files:
    latest = state_files[-1]
    state = json.loads(latest.read_text())
    print(f'Run: {latest.parent}')
    for k, v in state.items():
        print(f'  {k}: {v}')
else:
    print('No training output found yet.')

In [ ]:
# List all outputs
!find outputs -maxdepth 4 -name 'training_state.json' 2>/dev/null | sort || echo 'No outputs yet'

## Troubleshooting

**Colab disconnects:** Run `caffeinate -dims` on your laptop to keep the connection alive.

**Out of memory:** Use `--max_steps` to test with fewer steps, or pick a T4-optimized config.

**Disable W&B:**
```python
import os
os.environ['WANDB_MODE'] = 'disabled'
```